In [ ]:
# ---------------------------------------------
# High-quality quantile time series plots (R)
# Window selection: last/first n or explicit t-range
# ---------------------------------------------
suppressPackageStartupMessages({
  library(readr)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(purrr)
})

has_p <- function(df, p_target, tol = 1e-9) any(abs(df$p - p_target) < tol)
slice_p <- function(df, p_target, tol = 1e-9) dplyr::filter(df, abs(p - p_target) < tol)

derive_window_t <- function(all_t, n = NULL, side = c("last","first"), t_range = NULL, cap = 3300) {
  all_t <- sort(unique(as.numeric(all_t)))
  Tlen <- length(all_t)
  if (!is.null(t_range)) {
    tmin <- max(min(all_t), t_range[1])
    tmax <- min(max(all_t), t_range[2])
    return(all_t[all_t >= tmin & all_t <= tmax])
  }
  side <- match.arg(side)
  if (is.null(n)) n <- min(Tlen, cap)
  n <- max(1, min(n, Tlen))
  if (side == "last") tail(all_t, n) else head(all_t, n)
}

plot_series_with_quantiles <- function(csv_path,
                                       out_path = NULL,
                                       title    = NULL,
                                       add_bands = TRUE,
                                       n = NULL,             # how many t's to show
                                       side = c("last","first"),
                                       t_range = NULL,       # c(t_min, t_max) takes precedence over n/side
                                       cap = 3300) {

  stopifnot(file.exists(csv_path))
  df <- readr::read_csv(csv_path, show_col_types = FALSE) %>%
    mutate(t = as.numeric(t), p = as.numeric(p), q = as.numeric(q),
           y = as.numeric(y), mu = as.numeric(mu))

  # --- pick window of t's ---
  keep_t <- derive_window_t(df$t, n = n, side = side, t_range = t_range, cap = cap)

  # filter to window
  df_win <- df %>% filter(t %in% keep_t)

  obs <- df_win %>% distinct(t, y, mu) %>% arrange(t)
  qs  <- df_win %>% arrange(t, p)

  # optional bands
  bands <- list()
  if (add_bands && has_p(qs, 0.10) && has_p(qs, 0.90)) {
    band90 <- slice_p(qs, 0.10) %>% select(t, q_lo = q) %>%
      inner_join(slice_p(qs, 0.90) %>% select(t, q_hi = q), by = "t") %>%
      mutate(level = "90% band")
    bands[["90"]] <- band90
  }
  if (add_bands && has_p(qs, 0.25) && has_p(qs, 0.75)) {
    band50 <- slice_p(qs, 0.25) %>% select(t, q_lo = q) %>%
      inner_join(slice_p(qs, 0.75) %>% select(t, q_hi = q), by = "t") %>%
      mutate(level = "50% band")
    bands[["50"]] <- band50
  }
  bands_df <- if (length(bands)) bind_rows(bands) else NULL

  med <- if (has_p(qs, 0.5)) slice_p(qs, 0.5) else NULL

  # title/out path
  default_title <- paste0(basename(dirname(csv_path)), " — observations and quantile dynamics")
  base_title <- if (is.null(title)) default_title else title

  # describe window in subtitle
  tspan <- range(keep_t)
  subtitle <- paste0("t ∈ [", tspan[1], ", ", tspan[2], "] (", length(keep_t), " points)")

  p <- ggplot()

  if (!is.null(bands_df)) {
    p <- p + geom_ribbon(
      data = bands_df,
      aes(x = t, ymin = q_lo, ymax = q_hi, fill = level),
      alpha = 0.15
    )
  }

  p <- p + geom_line(
    data = qs, aes(x = t, y = q, group = p),
    linetype = "dashed", linewidth = 0.3, alpha = 0.6, color = "grey40"
  )

  if (!is.null(med)) {
    p <- p + geom_line(
      data = med, aes(x = t, y = q, linetype = "Median (p = 0.5)"),
      linewidth = 0.7, color = "black"
    )
  }

  p <- p +
    geom_line(data = obs, aes(x = t, y = y, color = "Observations y_t"),
              linewidth = 0.6, lineend = "round") +
    geom_point(data = obs, aes(x = t, y = y, color = "Observations y_t"),
               size = 0.8, stroke = 0.2) +
    labs(
      title = base_title, subtitle = subtitle,
      x = "Time (t)", y = "Value",
      color = NULL, linetype = NULL, fill = NULL
    ) +
    scale_x_continuous(expand = expansion(mult = c(0.01, 0.02))) +
    scale_y_continuous(expand = expansion(mult = c(0.05, 0.06))) +
    scale_linetype_manual(values = c("Median (p = 0.5)" = "solid")) +
    scale_fill_manual(values = c("90% band" = "grey60", "50% band" = "grey30")) +
    scale_color_manual(values = c("Observations y_t" = "#2C7BE5")) +
    guides(
      fill = guide_legend(order = 1, override.aes = list(alpha = 0.25)),
      linetype = guide_legend(order = 2),
      color = guide_legend(order = 3)
    ) +
    theme_minimal(base_size = 12) +
    theme(
      plot.title = element_text(face = "bold", hjust = 0, margin = margin(b = 4)),
      plot.subtitle = element_text(margin = margin(b = 8)),
      panel.grid.minor = element_blank(),
      panel.grid.major.x = element_line(linewidth = 0.2),
      panel.grid.major.y = element_line(linewidth = 0.2),
      legend.position = "top",
      legend.box = "vertical",
      legend.margin = margin(b = 0),
      axis.title.x = element_text(margin = margin(t = 6)),
      axis.title.y = element_text(margin = margin(r = 6))
    )

  if (is.null(out_path)) {
    stem <- basename(dirname(csv_path))
    out_path <- file.path(dirname(csv_path), paste0(stem, "_quantiles.png"))
  }

  ggsave(out_path, p, width = 10, height = 5.5, dpi = 300)
  message("Saved: ", out_path)
  invisible(p)
}

# --------------------------
# Run for your three series
# --------------------------
series_csvs <- c(
  "/data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_ar1V/series_long.csv",
  "/data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_bigW/series_long.csv",
  "/data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW/series_long.csv"
)

# EXAMPLES
# 1) Default: last min(T, 3300)
purrr::walk(series_csvs, ~ plot_series_with_quantiles(.x))

# 2) Last 100 points
purrr::walk(series_csvs, ~ plot_series_with_quantiles(.x, n = 50, side = "last"))

# 3) First 100 points
purrr::walk(series_csvs, ~ plot_series_with_quantiles(.x, n = 50, side = "first"))

# 4) Explicit time range (takes precedence)
# purrr::walk(series_csvs, ~ plot_series_with_quantiles(.x, t_range = c(200, 800)))


Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_ar1V/dlm_ar1V_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_bigW/dlm_constV_bigW_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW/dlm_constV_smallW_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_ar1V/dlm_ar1V_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_bigW/dlm_constV_bigW_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW/dlm_constV_smallW_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_ar1V/dlm_ar1V_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_bigW/dlm_constV_bigW_quantiles.png

Saved: /data/muscat_data/jaguir26/exdqlm/results/sim_suite_dlm/series/dlm_constV_smallW/dlm_constV